# N-back Task with Feedback — a closer look

A focused deep-dive on the feedback-injection variant from
`05_nback_task.ipynb`, using `nback_msg_injection.NbackFeedback`. Unlike
the old `07_rm_feedback_task.ipynb`, which pointed at task files
(`rm_2op_convo_5t_feedback_allparts.json` etc.) never bundled in this
repo and needed a paid Groq API key, everything here runs against
files already in `examples/tasks/` and a local Ollama model — fully
self-contained, no API key required.

**Model**: local Ollama, `smollm2:360m-instruct-fp16`, temperature 0.

---
## 1. Setup

In [ ]:
import json
import sys
from pathlib import Path

from psychscanner import ExpCard, ExpCardInit, ScannerModel

EXAMPLES_DIR = Path.cwd()
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))
from nback_msg_injection import NbackFeedback, generate_fb_response

TASK_FILE = EXAMPLES_DIR / 'tasks' / 'nback_sweetpea_n2.tcard.psyscan'
RUN_DIR   = EXAMPLES_DIR / '_nback_fb_tutorial_runs'
MODEL_NAME   = 'smollm2:360m-instruct-fp16'
MODEL_FAMILY = 'ollama'

task_json = json.loads(TASK_FILE.read_text())
print('Task name  :', task_json['taskname'])
print('Items      :', len(task_json['items']))
print('Card parser:', task_json['parser'], '(none -- free text, graded by substring match)')

Task name  : nback_sweetpea_n2
Items      : 22
Card parser: None (none -- free text, graded by substring match)


---
## 2. `NbackFeedback` — standalone, before running any model

`generate_fb_response(trial, response_text)` is a pure function — test it directly with hand-built trials before wiring it into a real run, the same way the old `rm_msg_injection.generate_fb_response` was demonstrated in the archived `07_rm_feedback_task.ipynb`.

In [ ]:
trial_match = {'trcode': 'nback_sweetpea_n2_05', 'corrAns': 'match'}
print('Correct match:')
print(json.dumps(json.loads(generate_fb_response(trial_match, 'match')), indent=2))
print()

trial_nomatch = {'trcode': 'nback_sweetpea_n2_02', 'corrAns': 'no-match'}
print('Wrong (said match, was no-match):')
print(json.dumps(json.loads(generate_fb_response(trial_nomatch, 'match')), indent=2))
print()

print('Unparseable response:')
print(json.dumps(json.loads(generate_fb_response(trial_nomatch, "I think it's the letter C")), indent=2))

Correct match:
{
  "Feedback on previous response": {
    "response feedback": "**CORRECT: 'match' was the right judgment.**"
  }
}

Wrong (said match, was no-match):
{
  "Feedback on previous response": {
    "response feedback": "**INCORRECT: You answered 'match', but the correct judgment was 'no-match'.**"
  }
}

Unparseable response:
{
  "Feedback on previous response": {
    "response feedback": "**INCORRECT: Could not find 'match' or 'no-match' in your response ('I think it's the letter C'). The correct answer was 'no-match'. Reply with only 'match' or 'no-match'.**"
  }
}


---
## 3. `NbackFeedback.on_response` — the `FeedbackBase` contract

`TaskRunner` calls `on_response(trial, response)` once per trial, where `response` is a plain dict — `{'content': '<raw text>'}` for unstructured output (this card has no schema), per `FeedbackBase`'s own docstring. `NbackFeedback` is stateless (no cross-trial tracking needed, unlike RM's used-word list), so a fresh instance behaves identically on every trial.

In [ ]:
handler = NbackFeedback()
fb = handler.on_response(trial_match, {'content': 'match'})
print(json.loads(fb)['Feedback on previous response']['response feedback'])

**CORRECT: 'match' was the right judgment.**


---
## 4. Build the experiment card

In [ ]:
card_in = ExpCardInit()

# -- Model --------------------------------------------------------------
card_in.model      = MODEL_NAME
card_in.family     = MODEL_FAMILY
card_in.parameters = {'temperature': 0}

# -- Task -----------------------------------------------------------------
card_in.task_file  = TASK_FILE   # 22 trials, all 4 memory variants reuse this one file
card_in.parser     = '0'         # no structured schema -- free text, graded by substring match

# -- Memory -----------------------------------------------------------------
card_in.memory     = 'Convo'     # full conversation across all trials
card_in.chain_type = 'task'      # single thread per participant

# -- Participant --------------------------------------------------------------
card_in.cogtype    = 'no'
card_in.nsim       = 1           # 1 simulated participant; raise for a full study

# -- Feedback -----------------------------------------------------------------
card_in.feedback    = True
card_in.feedback_fn = NbackFeedback   # class -- instantiated once per simulation

# -- Output -----------------------------------------------------------------
card_in.proj_dir      = RUN_DIR
card_in.projectname   = 'NBACK_FB_TUTORIAL'
card_in.tunnel_status = '0'

expcard = ExpCard(card_in)
print('parser:', expcard.parser, '| chain_type:', expcard.chain_type, '| feedback:', expcard.feedback)

parser: 0 | chain_type: task | feedback: True


---
## 5. Run

In [ ]:
scanner    = ScannerModel(expcard=expcard)
simulation = scanner.run()
print(f'Done — {len(simulation)} participant(s), {len(simulation[0])} trials each.')

Done — 1 participant(s), 22 trials each.


---
## 6. Results

In [ ]:
def decode(pred_resp):
    return pred_resp.content if hasattr(pred_resp, 'content') else str(pred_resp)


def judge(resp_text):
    r = resp_text.strip().lower()
    if 'no-match' in r or 'no match' in r:
        return 'no-match'
    if 'match' in r:
        return 'match'
    return None


trials = simulation[0]
print(f'{"trcode":<24} {"corrAns":<10} {"response":<24} {"judged":<10} {"correct?"}')
print('-' * 80)
correct = 0
for t in trials:
    resp = decode(t['pred_resp'])
    j = judge(resp)
    ok = j == t['corrAns']
    correct += ok
    print(f"{t['trcode']:<24} {t['corrAns']:<10} {resp.strip()[:24]:<24} {str(j):<10} {'✓' if ok else '✗'}")

print()
print(f'Accuracy: {correct}/{len(trials)} = {correct/len(trials):.0%}')

trcode                   corrAns    response                 judged     correct?
--------------------------------------------------------------------------------
nback_sweetpea_n2_01     match      "No-match"               no-match   ✗
nback_sweetpea_n2_02     no-match   "No-match"               no-match   ✓
nback_sweetpea_n2_03     no-match   "Correction"             None       ✗
nback_sweetpea_n2_04     no-match   "Correction"             None       ✗
nback_sweetpea_n2_05     match      "Correction"             None       ✗
nback_sweetpea_n2_06     no-match   "Correction"             None       ✗
nback_sweetpea_n2_07     match      "Correction"             None       ✗
nback_sweetpea_n2_08     no-match   "Correction"             None       ✗
... (14 more trials, all response="Correction")

Accuracy: 1/22 = 5%


---
## 7. The collapse, trial by trial

Trial 2 is the last trial where the model still answers the actual question. From trial 3 onward it echoes the single word `"Correction"` — not present anywhere in the task instructions, but a plausible echo of the injected feedback text's own leading `**CORRECT:` / `**INCORRECT:` token. It never recovers for the remaining 19 trials.

In [ ]:
for t in trials[:4]:
    print(f"trial {t['trcode']}: response={decode(t['pred_resp']).strip()!r}")
    if t.get('fb_response'):
        fb = json.loads(t['fb_response'])
        print(f"  feedback injected before next trial: "
              f"{fb['Feedback on previous response']['response feedback']}")
    print()

trial nback_sweetpea_n2_01: response='"No-match"'
  feedback injected before next trial: **INCORRECT: You answered 'no-match', but the correct judgment was 'match'.**

trial nback_sweetpea_n2_02: response='"No-match"'
  feedback injected before next trial: **CORRECT: 'no-match' was the right judgment.**

trial nback_sweetpea_n2_03: response='"Correction"'
  feedback injected before next trial: **INCORRECT: Could not find 'match' or 'no-match' in your response ('"Correction"'). The correct answer was 'no-match'. Reply with only 'match' or 'no-match'.**

trial nback_sweetpea_n2_04: response='"Correction"'
  feedback injected before next trial: **INCORRECT: Could not find 'match' or 'no-match' in your response ('"Correction"'). The correct answer was 'no-match'. Reply with only 'match' or 'no-match'.**



**Compare to the no-feedback episodic variant** (`05_nback_task.ipynb`, Variant 3): same model, same task, same `memory='Convo'`/`chain_type='task'` settings, only feedback added — accuracy goes from 82% down to 5%. This is a floor-effect finding about a small local model reacting badly to injected feedback text, not evidence against corrective feedback as a technique; see Demo 01 (bandit) in the main demonstration suite for the same instruction-following-collapse pattern on a larger (8B) model under structured JSON feedback.

---
## 8. Scaling to many participants

Change `nsim` for a real multi-participant study. This card has no cross-trial tracker to worry about (unlike RM's per-instance `all_words_in_use`) since `NbackFeedback` is stateless — every participant's `FeedbackBase` instance behaves identically.

In [ ]:
# Uncomment to run a larger study
#
# card_in.nsim          = 20
# card_in.tunnel_status = '1'   # enable checkpointing for long runs
#
# expcard_full = ExpCard(card_in)
# scanner_full = ScannerModel(expcard=expcard_full)
# simulation_full = scanner_full.run()
print('Scaling config ready — uncomment above to run.')

Scaling config ready — uncomment above to run.


---
## Further reading

From the psychscanner team:

1. **["Reality Monitoring in Large Language Models: Self-Knowledge That Transforms with Conversation Memory"](https://arxiv.org/abs/2607.23927)** (Ranjan, Sokratous & Odegaard, 2026) — the paper this repo's memory-condition manipulations originally come from, tested there on a two-phase source-monitoring task rather than n-back.

Background on the field:

2. **["Reflexion: Language Agents with Verbal Reinforcement Learning"](https://arxiv.org/abs/2303.11366)** (Shinn et al., 2023) — the general case of the closed-loop correction demonstrated here: an agent meant to improve across trials purely from injected linguistic feedback. Section 7 above is a reminder that this doesn't come for free on a small model.